# Crecimiento poblacional

## Objetivos

* Comprender el modelo de crecimiento poblacional con límite de capacidad de carga
* Identificar los parámetros que afectan el crecimiento poblacional
* Identificar las características y los métodos para resolver la ecuación diferencial
* Implementar una solución analítica de la ecuación diferencial y graficarla, en Python

## Teoría

Modelar el crecimiento poblacional es un problema importante que puede ser utilizado para entender la dinámica de poblaciones, el uso de recursos y la sostenibilidad. Para entender cómo se construye el modelo, se debe entender el problema de crecimiento poblacional sin límite de capacidad de carga. En este caso, la población tiene recursos ilimitados y puede crecer indefinidamente.

Para definir el crecimiento poblacional, la ecuación utiliza un factor de crecimiento constante $r$ multiplicado por el tamaño de la población.

Para obtener $r$ se puede utilizar un valor experimental donde después de varias observaciones se registra el valor de la población en diferentes tiempos y se calcula la tasa de crecimiento poblacional como $r=\frac{P(t+1)-P(t)}{P(t)}$. Por ejemplo, si se tiene la siguiente tabla de datos:

|Población en el tiempo $t$|Población en el tiempo $t+1$|Tasa de crecimiento poblacional|
|---|---|---|
|100|120|0.2|
|90|150|0.6667|
|120|100|-0.167|

La ecuación diferencial que describe este modelo es:

$$\frac{dP}{dt} = rP$$

Donde:
* $P$ es el tamaño de la población en el tiempo $t$
* $r$ es la tasa de crecimiento poblacional
* $t$ es el tiempo

Cuando la población tiene un límite ya sea por recursos o espacio, se dice que la población tiene un límite de capacidad de carga. Cuando la población está por debajo de la capacidad de carga, la población crece a una tasa positiva. Cuando la población está por encima de la capacidad de carga, la población decrece a una tasa negativa. Cuando la población está en el límite de capacidad de carga, la población no crece ni decrece. Para modelar este cambio se agrega un factor de amortiguamiento que depende del tamaño de la población y de la capacidad de carga.

Mientras la población esté más lejos de la capacidad de carga, el factor de amortiguamiento es mayor, por lo que la población crece o decrece más rápido. Mientras la población esté más cerca de la capacidad de carga, el factor de amortiguamiento es menor, por lo que la población crece o decrece más lento. Cuando la población está en el límite de capacidad de carga, el factor de amortiguamiento es cero, por lo que la población no crece ni decrece. En los siguientes datos se muestra cómo el factor de amortiguamiento cambia dependiendo del tamaño de la población y de la capacidad de carga:

|Población en el tiempo $t$|Capacidad de carga $K$|Factor de amortiguamiento $\left(1-\frac{P}{K}\right)$|
|---|---|---|
|300|200|-0.5|
|250|200|-0.25|
|210|200|-0.05|
|200|200|0.0|
|190|200|0.05|
|150|200|0.25|
|100|200|0.5|

La ecuación diferencial que describe este modelo es:

$$\frac{dP}{dt} = rP\left(1-\frac{P}{K}\right)$$

Donde:
* $P$ es el tamaño de la población en el tiempo $t$
* $r$ es la tasa de crecimiento poblacional
* $K$ es la capacidad de carga
* $t$ es el tiempo

## Implementación en Python

### Bibliotecas

En esta sección se importan las bibliotecas necesarias para trabajar con ecuaciones diferenciales, graficar y crear la interfaz interactiva.

In [ ]:
import sympy as sp # Cálculo simbólico
import numpy as np # Cálculo numérico optimizado

import matplotlib.pyplot as plt # Creación de gráficas
import matplotlib.colors as mcolors # Colores para el campo direccional

import ipywidgets as widgets # Hacer la interfaz interactiva
from IPython.display import display, Math, HTML # Hacer la interfaz interactiva

# Configuración para optimizar las gráficas en notebooks
%matplotlib inline

### Definición de variables simbólicas y la ecuación diferencial

En esta sección se definen las variables simbólicas con SymPy y se define la ecuación diferencial que describe el modelo de crecimiento poblacional con límite de capacidad de carga.

En la teoría ya definimos la ecuación diferencial que describe el crecimiento poblacional con límite de capacidad de carga.

$$\frac{dP}{dt} = rP\left(1-\frac{P}{K}\right)$$

Con esto podemos identificar las variables simbólicas que se necesitan:

|Variable|Descripción|Tipo en Sympy|
|---|---|---|
|$t$|Tiempo|Símbolo|
|$r$|Tasa de crecimiento poblacional|Símbolo|
|$K$|Capacidad de carga|Símbolo|
|$P$|Tamaño de la población en $t$|Función de $t$|
|$\frac{dP}{dt}$|Derivada de $P$ con respecto a $t$|Derivada de $P$ con respecto a $t$|

In [ ]:
t = sp.Symbol('t')
r = sp.Symbol('r')
K = sp.Symbol('K')
P = sp.Function('P')(t)
dPdt = sp.Derivative(P, t)

In [ ]:
ecuacion_diferencial = sp.Eq(dPdt, r * P * (1 - P / K))

### Funciones para analizar, resolver y graficar ecuaciones diferenciales

En esta sección se definen las funciones necesarias para analizar, resolver y graficar ecuaciones diferenciales. Estas funciones se pueden reutilizar para otros problemas de ecuaciones diferenciales.

In [ ]:
def analizar_ecuacion_diferencial(ecuacion, variable_dependiente):
    try:
        # Obtener los posibles métodos de solución
        posibles_metodos = sp.classify_ode(ecuacion, variable_dependiente)
    except:
        print("No se pudo clasificar la ecuación diferencial.")
        return

    # Buscar algun método que contenga la palabra 'linear' para saber si es lineal
    es_lineal = any('linear' in c for c in posibles_metodos)
    # Buscar algun método que contenga la palabra 'ordinary' para saber si es ordinaria
    es_ordinaria = any('ordinary' in c for c in posibles_metodos)
    # Obtener el orden de la ecuación diferencial
    orden = sp.ode_order(ecuacion, variable_dependiente)

    print("Ecuación diferencial planteada:")
    display(Math(sp.latex(ecuacion)))

    # Mostrar los resultados del analisis
    print("Clasificación  de la EDO:")
    print(f"Es lineal: {es_lineal}")
    print(f"Es ordinaria: {es_ordinaria}")
    print(f"Orden: {orden}")

    print("Posibles métodos de solución:")
    for metodo in posibles_metodos:
        print(f"- {metodo}")

In [ ]:
def obtener_solucion_general(ecuacion, variable_dependiente, metodo_definido=None):
    # Convertir decimales a racionales para evitar problemas de redondeos o residuos al hacer el cálculo simbólico
    ecuacion = ecuacion.replace(
        lambda x: x.is_Float,
        lambda x: sp.Rational(str(x))
    )

    try:
        # Si se proporciona un método específico, intentar resolver la ecuación diferencial con ese método
        if metodo_definido is not None:
            # Resolver la ecuación diferencial usando el método definido
            sol = sp.dsolve(
                ecuacion,
                func=variable_dependiente,
                hint=metodo_definido
            )
        else:
            # Si no se proporciona un método específico, dejar que sympy determine el mejor método automáticamente
            sol = sp.dsolve(
                ecuacion,
                func=variable_dependiente
            )
    except Exception as e:
        print(f"No se pudo resolver la ecuación diferencial: {e}")
        return None
    
    # En algunos casos se devuelve una lista de soluciones, en ese caso se toma la primera solución
    if isinstance(sol, list):
        solucion_general = sol[0]
    else:
        solucion_general = sol

    solucion_general = sp.simplify(solucion_general)
            
    # Redondear los decimales a 2 cifras significativas
    solucion_general = solucion_general.replace(
        lambda x: x.is_Float,
        lambda x: sp.Float(x, 2)
    )

    if solucion_general.lhs != variable_dependiente:
        solucion_general = sp.Eq(solucion_general.rhs, solucion_general.lhs)
            
    return solucion_general

In [ ]:
def obtener_solucion_particular(solucion_general, y0, variable_dependiente, variable_independiente):
    if solucion_general is None:
        return None

    # Si el lado izquierdo de la ecuación de la solución general no es igual a la variable dependiente, marcar error de formato
    if solucion_general.lhs != variable_dependiente:
        display(HTML("<p> Error: La solución general no tiene el formato esperado</p>"))
        return None

    # Obtener el lado derecho de la ecuación de la solución general
    expresion = solucion_general.rhs
    # Buscar el símbolo que se utilizó para representar la constante de integración
    C = next(iter(expresion.free_symbols - {variable_independiente}))

    # Sustituir la variable independiente por 0 y la variable dependiente por y0 para obtener la ecuación de la condición inicial
    condicion_inicial = sp.Eq(y0, expresion.subs(variable_independiente, 0))
    # Resolver la ecuacion para obtener el valor de la constante de integración
    soluciones_C = sp.solve(condicion_inicial, C, check=False)

    # Validar que se haya encontrado al menos una solución para la constante de integración
    if not soluciones_C:
        display(HTML("<p> Error: No se pudo encontrar una solución para la constante de integración.</p>"))
        return None
    # En algunos casos se devuelve una lista de soluciones, en ese caso se toma la primera solución
    if isinstance(soluciones_C, list):
        valor_C = soluciones_C[0]

    # Sustituir el valor de la constante de integración en la expresión de la solución general
    expresion_sustituida = expresion.subs(C, valor_C)
    # Simplificar la expresión resultante y redondear los decimales a 2 cifras significativas
    expresion_simplificada = sp.simplify(expresion_sustituida).replace(
        lambda x: x.is_Float,
        lambda x: round(float(x), 2)
    )

    # Construir la ecuación de la solución particular
    solucion_particular = sp.Eq(variable_dependiente, expresion_simplificada)

    return solucion_particular

In [ ]:
def graficar_campo_direccional(ecuacion, x_min, x_max, y_min, y_max, variable_dependiente, variable_independiente):    
    # Definir el número de puntos en cada eje para el campo direccional
    n_puntos = 30

    # Convertir la ecuación diferencial a una función que pueda ser evaluada numéricamente
    try:
        f = sp.lambdify((variable_independiente, variable_dependiente), ecuacion.rhs, modules='numpy')
    except Exception as e:
        return None

    # Crear la malla de puntos para el campo direccional
    valores_x = np.linspace(x_min, x_max, n_puntos)
    valores_y = np.linspace(y_min, y_max, n_puntos)
    X, Y = np.meshgrid(valores_x, valores_y)

    # Para graficar el campo direccional, se necesita calcular las componentes U y V del vector. v=(U, V) la pendiente de ese vector es V/U, 
    # la derivada representa la pendiente de la curva en ese punto, por lo que V/U=dy/dx=f(x, y). Para simplificar el calculo, se define U=1 y V=f(x, y)
    U = np.ones_like(X)
    V = f(X, Y)
    # Si algun valor de V es infinito o NaN, se reemplaza por NaN para evitar errores al graficar
    V = np.where(np.isfinite(V), V, np.nan)

    # Se obtiene la magnitud de cada vector, en caso de que la magnitud sea 0, se reemplaza por 1 para evitar división por 0
    magnitud = np.sqrt(U**2 + V**2)
    magnitud = np.where(magnitud == 0, 1, magnitud)

    # Se normalizan las componentes U y V para que todos los vectores tengan la misma longitud
    U_normalizada = U / magnitud
    V_normalizada = V / magnitud

    # Se calcula el ángulo de cada vector y se ajusta para que esté en el rango [0, 2π). El ángulo se utiliza para asignar un color a cada vector
    angulo = np.arctan2(V_normalizada, U_normalizada)
    angulo = (angulo + 2 * np.pi) % (2 * np.pi)

    # Se define la normalización de colores para que el mapa de colores se ajuste al rango de ángulos [0, 2π)
    normalizacion_colores = mcolors.Normalize(vmin=0, vmax=2*np.pi)

    # Crear la gráfica del campo direccional utilizando quiver, que dibuja vectores en cada punto de la malla
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.quiver(X, Y, U_normalizada, V_normalizada, angulo, cmap='RdBu', norm=normalizacion_colores, pivot='mid', scale=50)
    # Dar formato a la gráfica
    ax.set_title("Campo direccional de la ecuación diferencial del crecimiento poblacional")
    ax.set_xlabel(f"{variable_independiente}")
    ax.set_ylabel(f"{variable_dependiente}")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.close(fig)

    return fig

In [ ]:
def graficar_solucion_particular(solucion_particular, x_min, x_max, variable_dependiente, variable_independiente):
    if solucion_particular is None:
        return None

    # Si el lado izquierdo de la ecuación de la solución particular no es igual a la variable dependiente, marcar error de formato
    if solucion_particular.lhs != variable_dependiente:
        display(Math(r"\text{Error: La solución particular no está en el formato esperado.}"))
        return None

    # Definir el número de puntos para graficar la solución particular
    n_puntos = 200

    # Convertir la ecuación diferencial a una función que pueda ser evaluada numéricamente
    try:
        f = sp.lambdify(variable_independiente, solucion_particular.rhs, modules='numpy')
    except Exception as e:
        return None

    # Crear los valores de x para evaluar la solucion
    valores_x = np.linspace(x_min, x_max, n_puntos)
    # Evaluar la solución
    if variable_independiente in solucion_particular.rhs.free_symbols:
        # La solucion particular depende de la variable independiente
        valores_y = f(valores_x)
    else:
        # La solución particular es constante
        valor_constante = f(valores_x[0])
        valores_y = np.full(n_puntos, valor_constante)

    # Si algun valor de y es infinito o NaN, se reemplaza por NaN para evitar errores al graficar
    valores_y = np.where(np.isfinite(valores_y), valores_y, np.nan)

    # Encontrar el valor mínimo y máximo de y
    y_min = np.nanmin(valores_y)
    y_max = np.nanmax(valores_y)
    # Definir los valores que se mostraran en el eje y, se crean 11 valores equidistantes entre el valor mínimo y máximo de y
    # Con esto cada segmento del eje y representa 10% del rango de valores de y
    eje_y = np.linspace(y_min, y_max, 11)

    # Encontrar el valor mínimo y máximo de x
    x_min = np.nanmin(valores_x)
    x_max = np.nanmax(valores_x)
    # Definir los valores que se mostraran en el eje x, se crean 11 valores equidistantes entre el valor mínimo y máximo de x
    # Con esto cada segmento del eje x representa 10% del rango de valores de x
    eje_x = np.linspace(x_min, x_max, 11)

    # Crear la gráfica de la solución particular
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(valores_x, valores_y, color='blue', label='Solución particular')
    # Dar formato a la gráfica 
    ax.set_title("Solución particular de la ecuación diferencial del crecimiento poblacional")
    ax.set_xlabel(f"{variable_independiente}")
    ax.set_ylabel(f"{variable_dependiente}")
    ax.set_xlim(x_min, x_max)
    ax.set_yticks(eje_y)
    ax.set_xticks(eje_x)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.close(fig)
    
    return fig

## Laboratorio

### Creación de la interfaz

En esta sección se desarrolla la interfaz interactiva junto con todos los componentes necesarios para mostrar los resultados. Se incorporan sliders para modificar los parámetros de la ecuación diferencial y elementos para visualizar los resultados.

In [ ]:
# En esta celda se crean los sliders para los parámetros de la ecuación diferencial
slider_r = widgets.FloatSlider(
    value=0.1,
    min=-3.0,
    max=3.0,
    step=0.1,
    description='r:',
    continuous_update=False
)

slider_K = widgets.IntSlider(
    value=100,
    min=0,
    max=200,
    step=10,
    description='K:',
    continuous_update=False
)

slider_y0 = widgets.IntSlider(
    value=10,
    min=0,
    max=400,
    step=10,
    description='P(0):',
    continuous_update=False
)

slider_t_max = widgets.IntSlider(
    value=60,
    min=5,
    max=120,
    step=5,
    description='Rango de tiempo graficado:',
    continuous_update=False,
    style={'description_width': '200px'},
    layout=widgets.Layout(width='450px')
)

In [ ]:
# En esta celda se crean los componentes para mostrar los resultados
salida_modelo_sin_parametros = widgets.Output()
salida_analisis = widgets.Output()
salida_solucion_general = widgets.Output()
salida_solucion_particular = widgets.Output()
salida_campo_de_direccion = widgets.Output()
salida_grafica_solucion = widgets.Output()

In [ ]:
# En esta celda se configural los componentes que no cambian
with salida_modelo_sin_parametros:
    display(Math(sp.latex(ecuacion_diferencial)))

In [ ]:
# En esta celda se define la estructura de la interfaz interactiva, organizando los sliders y las salidas en filas y columnas
controles_parametros = widgets.HBox([
    slider_r,
    slider_K,
    slider_y0
])

controles_grafica = widgets.HBox([
    slider_t_max
])

controles = widgets.VBox([
    controles_parametros,
    controles_grafica
])

panel_analisis = widgets.VBox([
    widgets.HTML("<h3>Ecuación del modelo sin parámetros</h3>"),
    salida_modelo_sin_parametros,
    widgets.HTML("<h3>Análisis de la ecuación</h3>"),
    salida_analisis
])

panel_general_campo = widgets.VBox([
    widgets.HTML("<h3>Solución general</h3>"),
    salida_solucion_general,

    widgets.HTML("<h3>Campo direccional</h3>"),
    salida_campo_de_direccion
])

panel_particular_grafica = widgets.VBox([
    widgets.HTML("<h3>Solución particular</h3>"),
    salida_solucion_particular,

    widgets.HTML("<h3>Gráfica de la solución</h3>"),
    salida_grafica_solucion
])

resultados = widgets.GridBox(
    children=[
        panel_analisis,
        panel_general_campo,
        panel_particular_grafica
    ],
    layout=widgets.Layout(
        width='100%',
        grid_template_columns='20% 39% 39%',
        grid_template_rows='auto',
        grid_gap='1%'
    )
)

interfaz = widgets.VBox([
    widgets.HTML("<h1>Laboratorio de Crecimiento Poblacional</h1>"),
    controles,
    resultados
])

In [ ]:
def actualizar_interfaz(change=None): 
    # Obtener los valores de los sliders
    valor_r = slider_r.value 
    valor_K = slider_K.value 
    valor_y0 = slider_y0.value 
    valor_t_max = slider_t_max.value 

    # Sustituir los valores de los parámetros r y K en la ecuación diferencial
    ecuacion_con_parametros = ecuacion_diferencial.subs({ 
        r: valor_r, 
        K: valor_K 
    }) 

    # Limpiar las salidas de los resultados anteriores
    salida_analisis.clear_output() 
    salida_solucion_general.clear_output() 
    salida_campo_de_direccion.clear_output() 
    salida_solucion_particular.clear_output() 
    salida_grafica_solucion.clear_output() 

    # Analizar la ecuación diferencial con los parámetros sustituidos y mostrar los resultados
    with salida_analisis: 
        analizar_ecuacion_diferencial(ecuacion_con_parametros, P) 

    # Graficar el campo direccional de la ecuación diferencial con los parámetros sustituidos y mostrar la gráfica
    fig_campo = graficar_campo_direccional(ecuacion_con_parametros, 0, valor_t_max, 0, valor_K * 2, P, t) 
    with salida_campo_de_direccion: 
        display(fig_campo) 

    # Obtener la solución general de la ecuación diferencial con los parámetros sustituidos y mostrarla
    solucion_general  = obtener_solucion_general(ecuacion_con_parametros, P, "Bernoulli") 
    with salida_solucion_general: 
        display(Math(sp.latex(solucion_general))) 

    # Obtener la solución particular de la ecuación diferencial con los parámetros sustituidos y mostrarla
    solucion_particular = obtener_solucion_particular(solucion_general, valor_y0, P, t) 
    with salida_solucion_particular: 
        display(Math(sp.latex(solucion_particular))) 

    # Graficar la solución particular de la ecuación diferencial con los parámetros sustituidos y mostrar la gráfica
    fig_solucion = graficar_solucion_particular(solucion_particular, 0, valor_t_max, P, t) 
    with salida_grafica_solucion: 
        display(fig_solucion) 

In [ ]:
# Configurar los sliders para que llamen a la función actualizar_interfaz cuando su valor cambie
slider_r.observe(actualizar_interfaz, names='value')
slider_K.observe(actualizar_interfaz, names='value')
slider_y0.observe(actualizar_interfaz, names='value')
slider_t_max.observe(actualizar_interfaz, names='value')

### Laboratorio

Ejecuta todas las celdas anteriores para poder utilizar la interfaz interactiva. Despues ejecuta la siguiente celda para mostrar la interfaz interactiva. Para cambiar los parámetros no es necesario volver a ejecutar la celda, solo se deben mover los sliders y cambiar los valores de los parámetros.

In [ ]:
# Mostrar la interfaz interactiva y actualizarla por primera vez para mostrar los resultados iniciales
display(interfaz)
actualizar_interfaz()

### Ejercicios

1. ¿Qué predice el modelo cuando la configuración es $K = 0$, $r \neq 0$ y $P(0) \neq 0$? ¿Por qué?
<details>
<summary>Respuesta</summary>

No se puede generar un modelo. Al sustituir $K = 0$ en la ecuación diferencial, se obtiene una división por cero, lo que significa que el modelo no es válido matemáticamente. Si se interpreta esa configuracion se esta diciendo que el sistema no tiene capacidad de carga, por lo que la población no puede existir.

</details>

2. ¿Qué predice el modelo cuando la configuración es $K \neq 0$ y $P(0) = 0$? ¿Por qué?
<details>
<summary>Respuesta</summary>

No se puede generar un modelo. No existe población inicial, por lo que es imposible que crezca o decrezca algo que no existe; no importa que valor de $r$ o de $K$ se utilice.

</details>

3. ¿Qué predice el modelo cuando la configuración es $K \neq 0$, $r \neq 0$ y $P(0) = K$. ¿Por qué?
<details>
<summary>Respuesta</summary>

El modelo es una constante. El sistema alcanzo el límite de capacidad de carga, por lo que la población no crece ni decrece.

</details>

4. ¿Utilizando el campo direccional explica que predice el modelo cuando la configuración es $K \neq 0$ y $r = 0$. 
<details>
<summary>Respuesta</summary>

Todas las pendientes son cero, por lo que la población no crece ni decrece. Esto quiere decir que la poblacion sera estable para cualquier punto en el tiempo y cualquier condicion inicial.

</details>

5. Configura el laboratorio con los siguientes valores: $r = 0.1$, $K = 200$, $P(0) = 10$ y $\text{Rango de tiempo graficado} = 110$. En ese punto ya parece que se alcanzó el límite de capacidad de carga. Cada eje tiene 10 segmentos, por lo que cada segmento representa el 10% del rango de valores. ¿Qué porcentaje del rango de valores de $P$ se obtuvo para el 10, 20, 30, 40, 50, 60, 70, 80, 90 y 100% del tiempo graficado?
<details>
<summary>Respuesta</summary>

| Porcentaje del tiempo graficado | Porcentaje del rango de valores de $P$ |
|---|---:|
| 10% | 10% |
| 20% | 30% |
| 30% | ≈60% |
| 40% | 80% |
| 50% | ≈90% |
| 60% | ≈95% |
| 70% | ≈100% |
| 80% | ≈100% |
| 90% | ≈100% |
| 100% | ≈100% |

</details>

5. Con los resultados del ejercicio anterior, explica por qué se alcanza cerca del 95% del rango de valores de $P$ en el 60% del tiempo graficado, pero para alcanzar el 5% restante se necesita el 40% restante del tiempo graficado.
<details>
<summary>Respuesta</summary>

La solución de las ecuaciones diferenciales de primer orden es una curva exponencial, por lo que al inicio la población crece muy rápido y después de un tiempo el crecimiento se vuelve más lento.

</details>

6. Cambia los parámetros a los siguientes valores: $r = 0.1$, $K = 20$, $P(0) = 10$ y $\text{Rango de tiempo graficado} = 100$. En ese tiempo se alcanzó el límite de capacidad de carga. Si se duplica el valor de $r$, ¿Se alcanza el límite de capacidad de carga en la mitad del tiempo? Al ser una curva exponencial, se tiene una relación entre el tiempo y la tasa de crecimiento poblacional. ¿Cuál es esa relación?
<details>
<summary>Respuesta</summary>

Si, al duplicar el valor de $r$ se alcanza el límite de capacidad de carga en la mitad del tiempo. La relación entre $r$ y el tiempo es inversamente proporcional.

</details>

7. En los ejercicios anteriores, al graficar la solución particular, el límite superior de la gráfica es igual a la capacidad de carga $K$. Evalúa en una calculadora la solución particular con $t = \text{Rango de tiempo graficado}$. ¿El resultado es igual a $K$? Si no es igual, explica por qué.
<details>
<summary>Respuesta</summary>

No, el resultado no es igual a $K$, ya que la solución particular es una curva exponencial que se aproxima a $K$, pero nunca lo alcanza. Al graficar, lo que pasa es que se redondea el valor de la solución particular y se muestra como si fuera igual a $K$. Para corroborar que no existe ningún valor $t$ que haga que la solución particular sea igual a $K$, se puede resolver la solución particular para $P(t) = K$.
$$ 20 = \frac{20e^{\frac{t}{10}}}{e^{\frac{t}{10}} + 1} $$
No existe ningún valor de $t$ que satisfaga la ecuación, por lo que la solución particular nunca es igual a $K$.

</details>

8. Configura el laboratorio con los siguientes valores: $r = 0.1$, $K = 20$, $P(0) = 10$ y $\text{Rango de tiempo graficado} = 120$. Observa unicamente el campo direccional, ¿A qué valor de $P$ tienden todos los vectores del campo direccional? ¿Que significa que algunos vectores sean de color azul y otros de color rojo? ¿Como es la pendiente de los vectores debajo y arriba del valor al que tienden todos los vectores?
<details>
<summary>Respuesta</summary>

Todos los vectores del campo direccional tienden a $P = 20$ esto es la capacidad de carga del sistema. Los vectores de color azul representan que la población decrece (pendiente negativa) y los vectores de color rojo representan que la población crece (pendiente positiva). Los vectores debajo de $P = 20$ tienen pendiente positiva y los vectores arriba de $P = 20$ tienen pendiente negativa.

</details>

9. Modifica el valor de $r = -0.1$. Observa unicamente el campo direccional, ¿Se modifico el valor al que tienden todos los vectores del campo direccional? ¿Como es la pendiente de los vectores debajo y arriba de $P = 20$? ¿Como es la pendiente de los vectores cuando $P = 0$?
<details>
<summary>Respuesta</summary>

Si, los vectores tienden a $P = 0$ y $P = \infty$. Los vectores debajo de $P = 20$ tienen pendiente negativa y los vectores arriba de $P = 20$ tienen pendiente positiva. Los vectores cuando $P = 0$ tienen pendiente igual a cero.

</details>

10. Interpreta el modelo para los siguientes casos: $0 < P(0) < K$, $P(0) = K$ y $P(0) > K$. ¿Como se interpreta tener un valor de $r < 0$? ¿Tiene sentido la direccion del campo direccional en cada caso?

<details>
<summary>Respuesta</summary>

Tener un valor de $r < 0$ significa que la población decrece el indice de mortalidad es mayor que el indice de natalidad. En el caso donde $0 < P(0) < K$, la población decrece hasta llegar a $P = 0$, se detiene en ese punto ya que no se puede tener una población negativa; **el modelo tiene sentido**. En el caso donde $P(0) = K$, la población se mantiene constante en $P = K$; **el modelo no tiene sentido** ya que la población debería decrecer hasta llegar a $P = 0$. En el caso donde $P(0) > K$, la población crece sin un limite $P \to \infty$; **el modelo no tiene sentido** ya que la población debería decrecer hasta llegar a $P = 0$.

</details>

Extra. Utilizando el codigo de este laboratorio crea un nuevo laboratorio donde el modelo de crecimiento poblacional tenga sentido para valores de $r < 0$ para todos los casos de $P(0)$ es decir que la población decrezca hasta llegar a cero $P \to 0$.

## Referencias

Zill, D. G., & Cullen, M. (2009). Ecuaciones diferenciales con problemas de valores en la frontera. Cengage Learning Editores.